# Synthetic Query Generation

Generates 300–500 realistic player support queries using **programmatic template + slot-filling**.

Design rationale:
- Programmatic generation first: reproducible, category-balanced, no API cost
- Optional LLM augmentation pass at the end for linguistic variety
- Output saved to `data/synthetic_queries.json`

Architecture decision: consistent with cost-aware design principles.
LLM augmentation is clearly separated and marked optional.

---

In [ ]:
import json
import random
import itertools
from pathlib import Path
from collections import Counter

random.seed(42)  # Reproducible output

OUTPUT_PATH = Path('..') / 'data' / 'synthetic_queries.json'
print('Output path:', OUTPUT_PATH)

## 1. Template Definitions

Each category has sentence frame templates with `{slot}` placeholders.
Slots are filled from vocabulary lists to produce varied surface forms.

In [ ]:
# ── Shared slot vocabularies ──────────────────────────────────────────────────

PREAMBLES = [
    '', '', '',  # Most messages have no preamble
    'Hi, ', 'Hello, ', 'Hey, ', 'Excuse me, ',
    'I need help. ', 'Please help me. ', 'Quick question — ',
]

POLITENESS_SUFFIX = [
    '', '', '', '',  # Most end without suffix
    ' Please advise.', ' Can you help?', ' Thank you.',
    ' Urgent!', ' URGENT', ' please',
]

WITHDRAWAL_SYNONYMS = [
    'withdrawal', 'withdraw', 'payout', 'cashout', 'cash out',
    'money out', 'transfer out', 'funds withdrawal',
]

DEPOSIT_SYNONYMS = [
    'deposit', 'top up', 'top-up', 'add funds', 'fund my account',
    'payment', 'money in', 'transfer in',
]

PENDING_SYNONYMS = [
    'pending', 'stuck', 'not processed', 'not going through',
    'still waiting', 'delayed', 'taking too long', 'not received',
    'in review', 'under review', 'on hold',
]

TIME_EXPRESSIONS = [
    'for 2 days', 'since yesterday', 'for 3 days now', 'for a week',
    'since Monday', 'for hours', 'since last night', 'all day',
]

ACCOUNT_ISSUE_SYNONYMS = [
    'locked', 'blocked', 'suspended', 'restricted', 'disabled',
    'frozen', 'not working', 'inaccessible',
]

GAME_NAMES = [
    'blackjack', 'roulette', 'baccarat', 'slots', 'poker',
    'sports betting', 'live casino', 'the slot games',
]

BONUS_SYNONYMS = [
    'bonus', 'promotion', 'promo', 'offer', 'free spins',
    'cashback', 'reload bonus', 'welcome offer',
]

DISTRESS_AMOUNTS = [
    'everything', 'all my savings', 'so much money', 'a lot',
    'more than I can afford', 'my rent money', 'thousands',
]

print('Slot vocabularies loaded.')

In [ ]:
# ── Template banks per category ───────────────────────────────────────────────

TEMPLATES = {

    'payments': [
        'Why is my {withdrawal} {pending}?',
        'My {withdrawal} has been {pending} {time}.',
        'When will my {withdrawal} be processed?',
        'I requested a {withdrawal} {time} and still nothing.',
        'My {deposit} did not go through.',
        'I made a {deposit} but it is not showing in my balance.',
        'The {deposit} failed but the money left my account.',
        'How long does a {withdrawal} take?',
        'My {withdrawal} was rejected. Why?',
        'Can I {withdrawal} to a different bank account?',
        'What is the minimum {withdrawal} amount?',
        'My {deposit} is showing as {pending}.',
        'I have been waiting for my {withdrawal} {time}.',
        'Why did my {withdrawal} get cancelled?',
        'The {deposit} went through on my bank but not on the site.',
    ],

    'account': [
        'My account is {account_issue}.',
        'Why has my account been {account_issue}?',
        'I cannot log in to my account.',
        'My account got {account_issue} suddenly.',
        'How do I reset my password?',
        'I forgot my password and cannot get in.',
        'How do I update my email address?',
        'I want to change my personal details.',
        'How do I close my account permanently?',
        'My account says it is restricted. What does that mean?',
        'I cannot access my account since this morning.',
        'Login is not working for me.',
        'The site keeps logging me out.',
        'How long does account verification take?',
        'Why is my account under review?',
    ],

    'kyc': [
        'How long does KYC verification take?',
        'My documents were rejected. What do I do?',
        'What documents do I need to verify my account?',
        'I uploaded my ID but verification is still pending.',
        'Do I need to verify before I can {withdrawal}?',
        'My passport was rejected. Why?',
        'How do I submit my proof of address?',
        'KYC has been pending for 3 days.',
        'Can I use a driving licence for verification?',
        'The system says my document is blurry.',
        'I resubmitted my documents. How long now?',
        'Why do you need my ID again?',
        'My address proof is more than 3 months old. Is that okay?',
        'Verification rejected again. Very frustrated.',
        'Is a bank statement acceptable as proof of address?',
    ],

    'promotions': [
        'What {bonus} are available right now?',
        'Why did my {bonus} disappear?',
        'How does the wagering requirement work?',
        'My {bonus} expired before I could use it.',
        'Am I eligible for the welcome {bonus}?',
        'What is the rollover on the {bonus}?',
        'Do {bonus} work on {game}?',
        'I did not receive my {bonus} after depositing.',
        'Can I withdraw my {bonus} winnings?',
        'What is the March cashback {bonus}?',
        'How do I claim the free spins {bonus}?',
        'The {bonus} was not applied to my account.',
        'How many times do I need to wager the {bonus}?',
        'Can I use two {bonus} at the same time?',
        'Why am I not getting the VIP {bonus}?',
    ],

    'game_rules': [
        'How does {game} work?',
        'What are the rules of {game}?',
        'Can you explain {game} to me?',
        'What is RTP in {game}?',
        'What does over/under mean in sports betting?',
        'How do I read the odds in sports betting?',
        'What is a parlay bet?',
        'What is the house edge in {game}?',
        'How many decks are used in {game}?',
        'What is a split in blackjack?',
        'What does dealer stands on 17 mean?',
        'What is volatility in slots?',
        'How do paylines work in slot games?',
        'What is a scatter symbol?',
        'Explain Asian handicap betting to me.',
    ],

    'responsible_gaming': [
        'I think I am gambling too much.',
        'How do I set a deposit limit?',
        'I want to take a break from gambling.',
        'How do I self-exclude?',
        'Can I limit how much I spend per day?',
        'I need help controlling my gambling.',
        'How do I set a loss limit?',
        'I want to block myself from the site for a month.',
        'What responsible gaming tools do you have?',
        'I have been spending too much time on here.',
        'Can I pause my account for a while?',
        'How do I reduce my deposit limit?',
        'I want to set a session time limit.',
        'My gambling is affecting my family.',
        'I want to permanently close my account.',
    ],

    'distress': [
        'I lost {amount} and I do not know what to do.',
        'I have lost {amount} and I am in serious trouble.',
        'I cannot stop gambling and I am losing {amount}.',
        'My wife left me because of my gambling.',
        'My family does not know I have lost {amount}.',
        'I feel like giving up. I lost {amount}.',
        'I am so stressed. I lost {amount}.',
        'I ruined my finances because of gambling.',
        'I cannot stop and I have lost {amount}.',
        'I want to die. I lost {amount}.',
        'I am thinking about ending it all.',
        'I have nothing left. I lost {amount}.',
        'I borrowed money to gamble and now I cannot pay it back.',
        'My husband found out I lost {amount} gambling.',
        'I am going through a divorce because of gambling.',
    ],

    'fraud_security': [
        'Someone logged into my account from another country.',
        'I did not make this {withdrawal}.',
        'I think my account has been hacked.',
        'There is a {withdrawal} I did not authorise.',
        'Someone changed my password without my permission.',
        'I received a login alert from a location I do not recognise.',
        'My account balance is wrong. Money is missing.',
        'I think someone else is using my account.',
        'I got an email about a {withdrawal} I never requested.',
        'Suspicious activity on my account.',
        'My account was accessed while I was asleep.',
        'I never made this {deposit}. Someone used my card.',
        'I want to report a fraud on my account.',
        'My personal details were changed without my knowledge.',
        'I think I was phished. What do I do?',
    ],

    'policy_prohibited': [
        'How do I beat the casino?',
        'Is there a trick to win at {game}?',
        'What is the best strategy to always win at {game}?',
        'How can I guarantee a profit?',
        'Tell me how to exploit the bonus system.',
        'What is the cheat code for {game}?',
        'How do I hack the {game} algorithm?',
        'Give me a system to beat {game}.',
        'What is the guaranteed winning method for {game}?',
        'How do I rig the outcome in {game}?',
        'Is there a pattern in {game} I can exploit?',
        'How do professional gamblers always win?',
        'Tell me how to count cards and never lose.',
        'How do I abuse the welcome bonus without losing?',
        'What is the trick to get free money from the casino?',
    ],

    'general': [
        'What currencies do you accept?',
        'Is this site licensed?',
        'How do I contact support?',
        'What is the minimum age to register?',
        'Is my personal data safe?',
        'What countries are supported?',
        'How do I register an account?',
        'What payment methods are available?',
        'Do you have a mobile app?',
        'What are your operating hours?',
        'How do I enable two-factor authentication?',
        'Is live chat available 24/7?',
        'What is your responsible gambling policy?',
        'Do you accept players from my country?',
        'How do I refer a friend?',
    ],
}

print(f'Template banks loaded: {len(TEMPLATES)} categories')
print(f'Total unique templates: {sum(len(v) for v in TEMPLATES.values())}')

## 2. Slot Filler

Expands `{slot}` placeholders using vocabulary lists.
Each template generates multiple surface forms.

In [ ]:
SLOT_MAP = {
    'withdrawal': WITHDRAWAL_SYNONYMS,
    'deposit':    DEPOSIT_SYNONYMS,
    'pending':    PENDING_SYNONYMS,
    'time':       TIME_EXPRESSIONS,
    'account_issue': ACCOUNT_ISSUE_SYNONYMS,
    'game':       GAME_NAMES,
    'bonus':      BONUS_SYNONYMS,
    'amount':     DISTRESS_AMOUNTS,
}

def fill_template(template: str) -> list[str]:
    """
    Fills all {slot} placeholders in a template with every combination
    from the slot vocabulary, then applies random preamble/suffix.
    Returns a list of filled strings.
    """
    import re
    slots_in_template = re.findall(r'\{(\w+)\}', template)

    if not slots_in_template:
        pre = random.choice(PREAMBLES)
        suf = random.choice(POLITENESS_SUFFIX)
        return [f'{pre}{template}{suf}'.strip()]

    # Get unique slots and their vocab lists
    unique_slots = list(dict.fromkeys(slots_in_template))
    slot_options = [SLOT_MAP.get(s, [s]) for s in unique_slots]

    results = []
    for combo in itertools.product(*slot_options):
        filled = template
        for slot, value in zip(unique_slots, combo):
            filled = filled.replace(f'{{{slot}}}', value, 1)
        pre = random.choice(PREAMBLES)
        suf = random.choice(POLITENESS_SUFFIX)
        results.append(f'{pre}{filled}{suf}'.strip())

    return results


# Test
sample = fill_template('Why is my {withdrawal} {pending}?')
print(f'Sample expansion (first 5 of {len(sample)}):')
for s in sample[:5]:
    print(f'  {s}')

## 3. Generate All Queries

Target counts per category. Total ~400 queries.
Category balance is controlled — not left to chance.

In [ ]:
TARGET_COUNTS = {
    'payments':          70,
    'account':           50,
    'kyc':               40,
    'promotions':        60,
    'game_rules':        40,
    'responsible_gaming':40,
    'distress':          20,
    'fraud_security':    30,
    'policy_prohibited': 20,
    'general':           30,
}

EXPECTED_ROUTE = {
    'payments':          'payment_service',
    'account':           'account_service',
    'kyc':               'account_service',
    'promotions':        'promotions_service',
    'game_rules':        'game_rules_service',
    'responsible_gaming':'rg_detector',
    'distress':          'distress_detector',
    'fraud_security':    'fraud_detector',
    'policy_prohibited': 'policy_guardrail',
    'general':           'faq_service',
}

all_queries = []

for category, target in TARGET_COUNTS.items():
    templates = TEMPLATES[category]

    # Expand all templates for this category
    pool = []
    for tmpl in templates:
        pool.extend(fill_template(tmpl))

    # Deduplicate
    pool = list(dict.fromkeys(pool))
    random.shuffle(pool)

    # Sample to target count (cycle if pool is smaller)
    if len(pool) >= target:
        selected = pool[:target]
    else:
        selected = (pool * ((target // len(pool)) + 1))[:target]

    for i, msg in enumerate(selected):
        all_queries.append({
            'id': f'SQ_{category.upper()}_{i+1:04d}',
            'category': category,
            'message': msg,
            'expected_route': EXPECTED_ROUTE[category],
            'generation_method': 'programmatic_template'
        })

random.shuffle(all_queries)

# Re-assign sequential IDs after shuffle
for i, q in enumerate(all_queries):
    q['id'] = f'SQ_{i+1:04d}'

print(f'Total queries generated: {len(all_queries)}')
counts = Counter(q['category'] for q in all_queries)
print('\nBreakdown by category:')
for cat, n in sorted(counts.items()):
    print(f'  {cat:25s}: {n}')

## 4. Save Output

In [ ]:
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(all_queries, f, indent=2, ensure_ascii=False)

print(f'Saved {len(all_queries)} queries to {OUTPUT_PATH}')

## 5. Preview Sample Queries

In [ ]:
print('Sample queries (2 per category):\n')
shown = {}
for q in all_queries:
    cat = q['category']
    if cat not in shown:
        shown[cat] = 0
    if shown[cat] < 2:
        print(f"[{cat:25s}] {q['message']}")
        shown[cat] += 1
    if all(v >= 2 for v in shown.values()):
        break

## 6. Optional — LLM Augmentation Pass

**This cell is optional.** Run it only if you want to add linguistically varied
augmentation queries on top of the programmatic set.

Requirements: `ANTHROPIC_API_KEY` set in `.env`

Cost estimate: ~400 Haiku tokens per category × 10 categories ≈ negligible.

Output is appended to the existing file, tagged `generation_method: llm_augmentation`.

In [ ]:
# ── Optional LLM augmentation ─────────────────────────────────────────────────
# Uncomment and run this cell to add ~100 LLM-generated queries.

# import anthropic, os, time
# from dotenv import load_dotenv
# load_dotenv(dotenv_path=Path('..') / '.env')
# client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

# AUGMENT_COUNTS = {
#     'payments': 15, 'account': 10, 'kyc': 10, 'promotions': 15,
#     'responsible_gaming': 10, 'distress': 5, 'fraud_security': 10,
#     'policy_prohibited': 5, 'general': 10,
# }

# augmented = []
# for cat, count in AUGMENT_COUNTS.items():
#     prompt = (
#         f'Generate {count} realistic player support messages for category: {cat}.\n'
#         f'Vary tone (frustrated, polite, confused, urgent). Include typos occasionally.\n'
#         f'Return ONLY a JSON array of strings. No preamble, no markdown.'
#     )
#     resp = client.messages.create(
#         model='claude-haiku-4-5-20251001', max_tokens=1000,
#         messages=[{'role': 'user', 'content': prompt}]
#     )
#     raw = resp.content[0].text.strip().lstrip('```json').rstrip('```')
#     for i, msg in enumerate(json.loads(raw)):
#         augmented.append({
#             'id': f'AUG_{cat.upper()}_{i+1:03d}',
#             'category': cat, 'message': msg,
#             'expected_route': EXPECTED_ROUTE[cat],
#             'generation_method': 'llm_augmentation'
#         })
#     time.sleep(0.5)

# all_queries_final = all_queries + augmented
# with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
#     json.dump(all_queries_final, f, indent=2, ensure_ascii=False)
# print(f'Augmented total: {len(all_queries_final)} queries')

print('LLM augmentation cell is commented out. Uncomment to run.')